In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx

trajectory_segements = gpd.read_file("../../data/processed/segments.gpkg")

clean_trajectories = []
valid_indices = []

THRESH = 1000
MIN_SIZE =  5


for idx, geom in enumerate(trajectory_segements.geometry):

    if geom is None or geom.is_empty:
        continue

    coords = list(geom.coords)

    if len(coords) < 2:
        continue

    coords_2d = [(c[0], c[1]) for c in coords]
    arr = np.asarray(coords_2d, dtype=np.float64)

    if arr.ndim != 2 or arr.shape[1] != 2:
        continue

    if not np.isfinite(arr).all():
        continue

    clean_trajectories.append(arr)
    valid_indices.append(idx)

trajectory_segements = trajectory_segements.iloc[valid_indices].reset_index(drop=True)
trajectories = clean_trajectories



In [ ]:

results = pd.read_parquet(("../../data/processed/frechet_results.parquet"))


In [ ]:
G = nx.Graph()
G.add_nodes_from(range(N))

for row in results.itertuples(index=False):
    i, j, d = row

    # ensure indices are valid
    if i < N and j < N:
        if d < THRESH:
            G.add_edge(i, j, weight=d)



In [ ]:
clusters = list(nx.connected_components(G))

valid_clusters = [c for c in clusters if len(c) >= MIN_SIZE]


labels = np.full(N, -1, dtype=int)

for cluster_id, cluster in enumerate(valid_clusters):
    for i in cluster:
        labels[i] = cluster_id

trajectory_segements["cluster"] = labels




In [ ]:
gdf_corridors = trajectory_segements[trajectory_segements["cluster"] != -1].copy()


gdf_corridors.to_file(
    "../../data/processed/trajectory_clusters.gpkg",
    layer="corridors",
    driver="GPKG"
)
